## STS & cross-account access

**STS** — the Security Token Service — is the engine that mints **temporary credentials** whenever a role is assumed. It returns an access key ID, a secret access key, a **session token**, and an expiration between **15 minutes and 12 hours**. After that the credentials simply stop working — which is why roles are safer than long-lived keys.

There are five `AssumeRole` flavors: plain **`AssumeRole`** (same- or cross-account), **`AssumeRoleWithSAML`** (enterprise IdPs — ADFS, Okta, Ping), **`AssumeRoleWithWebIdentity`** (OIDC — Cognito, Google, EKS, GitHub Actions), **`GetSessionToken`** (wrap a user's keys with MFA), and **`GetFederationToken`** (a custom broker). On EC2 an **instance profile** wraps a role and the metadata service vends rotating creds automatically; Lambda execution roles and ECS task roles do the same — so you **never plant a long-lived key on a box.**

**Cross-account access** has two shapes. With **role assumption**, the resource account creates a role trusting the caller's account; the caller calls `sts:AssumeRole` and *becomes* the role — their original identity is dropped. With a **resource-based policy**, the resource account names the caller's principal directly in the resource's policy, and the caller *keeps* their identity end to end. One guard rail matters when a **third-party vendor** assumes a role in your account: the **confused-deputy** problem. Require an **`sts:ExternalId`** (a secret only you and the vendor share) so another of the vendor's customers can't trick them into reaching into your account; for an AWS *service* calling on your behalf, use **`aws:SourceArn` / `aws:SourceAccount`** instead.